## 🔰PyTorchでニューラルネットワーク基礎 #29 【MLM事前学習編・マスク化】

BERTタイプのMLM事前学習で利用するマスク化処理の部分

### 内容
* Qiitaの記事と連動しています

### データmlm_pretrain_sample.jsonについて
* text：日本語文
* ids：適当なトークナイザーでID化したもの。等長ベクトルになっている

### ちょっとした注意点
* TensorDatasetとDataLoaderのパターンで利用するタイプ

### 2.1 mlm用のcollate関数の引数を確認

In [2]:
import torch
from torch.utils.data import TensorDataset  
import json

# (1) データ読込と必要な部分の切り出し
with open("./data/mlm_pretrain_sample.json", "r") as f:
    data = json.load(f)
# x: IDベクトル
x = torch.LongTensor([item["ids"] for item in data])

# (2) TensorDataset
data = TensorDataset(x)
print(data.__getitem__(0))

(tensor([   1,  332, 8061, 8716, 8267, 8304, 8599, 7615, 4521, 9287, 7486,  211,
        7722, 7822, 4178, 3913, 8022, 8392, 8263, 3932, 3749,    2,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0]),)


### 2.2 mlm用のcollate関数を作成してみよう

In [3]:
import random

def mlm_sample(batch, mask_prob:float = 0.15):
    """
    batch: [(x1,), (x2,),...,(xi,),...] というタプルのリストの予定
    x1: tensor
    """

    special_tokens = [0,1,2,3,4]  # <pad>, <bos>, <eos>, <unk>, <mask>になる予定
    normal_tokens = range(5,10_000) # 語彙IDのspecial_tokensを除いたもの　語彙数10k
    
    all_input_ids = []   # <mask>化したID列
    all_labels = []      # 15%の予測部分に正解のIDを、それ以外は　−100　で埋める

    # (1) 各サンプルに対してループ処理
    for item in batch:
        original_ids = item[0]                        # item[0]でxiを取り出す
        input_ids = original_ids.clone()              # originalも使うのでcloneしておく
        labels = torch.full_like(original_ids, -100)  # ラベル：一旦　ID = -100 にしてから、該当部分のみ書き換える
       
        # (2) マスク化
        for i in range(len(original_ids)):

            # (3) 特殊トークンの場合は飛ばす
            token_id = original_ids[i]
            if token_id in special_tokens:
                continue

            # (4) 特殊トークン以外15%の確率が置き換え対象
            if random.random() < mask_prob:
                labels[i] = original_ids[i]   # labelsに元のIDを入れる (教師データになる)
                # 置き換え対象の処理 80/10/10 で マスク/他トークン/そのまま
                rand = random.random()
                if rand < 0.8:
                    input_ids[i] = 4 # mask_id 今回はそのまま入力しているけど
                elif rand < 0.9:
                    input_ids[i] = random.choice(normal_tokens) # 特殊トークンID以外のトークンで置き換え
                else:
                    pass # 10%はそのまま
        
        all_input_ids.append(input_ids)
        all_labels.append(labels)

    # (5) リストをスタックして (bs, seq_Len) のテンソルに戻す
    return torch.stack(all_input_ids), torch.stack(all_labels)

### 2.3 動作確認

In [26]:
org_ids = torch.LongTensor([1, 219, 338, 295, 4188, 6988, 4451, 4218, 4057, 4464, 441, 9127, 4252, 321, 317, 4165, 4369, 160, 2])
x1 = org_ids.clone()
batch = [(x1,)]

# 30%置き換え対象になっています
input_ids, labels = mlm_sample(batch, mask_prob=0.3)

# くどいプリント連打😅
print("変更されたID列")
print(f"{input_ids=}")
print(f"{input_ids.shape=}")
print("対応するラベル")
print(f"{labels=}")
print(f"{labels.shape=}")

print("変換部分を表示")
ids = torch.where(input_ids[0] != org_ids)

for num in ids[0]:
    print(f"input: {input_ids[0][num].item()},\t label: {labels[0][num]}")

変更されたID列
input_ids=tensor([[   1,  219, 4978,  295,    4, 6988,    4, 4218, 4057, 4464,  441, 9127,
         4252,    4,  317, 4165, 4369,  160,    2]])
input_ids.shape=torch.Size([1, 19])
対応するラベル
labels=tensor([[-100, -100,  338, -100, 4188, -100, 4451, -100, -100, -100, -100, -100,
         -100,  321, -100, -100, -100, -100, -100]])
labels.shape=torch.Size([1, 19])
変換部分を表示
input: 4978,	 label: 338
input: 4,	 label: 4188
input: 4,	 label: 4451
input: 4,	 label: 321


## 3. データ読み込みから学習ループまでの流れ

In [28]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import json
import random

# deviceの指定
device = "cuda" if torch.cuda.is_available() else "cpu"

# (1) mlm_sample関数　（省略）
#def mlm_sample(batch, mask_prob:float = 0.15):
#    """ 省略 """
#    return torch.stack(all_input_ids), torch.stack(all_labels)

# (2) サンプルデータ
with open("./data/mlm_pretrain_sample.json", "r") as f:
    data = json.load(f)
x = torch.LongTensor([item["ids"] for item in data])

# (3) TensorDatasetにする
train_data = TensorDataset(x)

# (4) DataLoaderを使う
train_loader = DataLoader(
    train_data, 
    batch_size=100, 
    shuffle=True, 
    collate_fn=mlm_sample,   # ここで作成した関数を指定
    num_workers=4
)

# (5) 学習ループ
for epoch in range(2):
    for x, t in train_loader:
        x = x.to(device)
        t = t.to(device)
        print(f"{epoch}: x: {x.shape},\tt: {t.shape}")
        print(f"x[0]:\n{x[0]}")
        print(f"t[0]:\n{t[0]}\n")
        break

0: x: torch.Size([100, 64]),	t: torch.Size([100, 64])
x[0]:
tensor([   1, 1465, 4157, 4301, 3879, 4077, 9539,  210, 3639,  199, 3942, 5289,
           4, 8204,  210, 8743,    4, 8488, 3749,    2,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0], device='cuda:0')
t[0]:
tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, 5069, -100, -100,
        4608, -100, -100, -100, 7946, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100], device='cuda:0')

1: x: torch.Size([100, 64]),	t: t

### 他の確認方法

In [29]:
# batch は collate_fn (create_mlm_sample) の戻り値のタプル
batch = next(iter(train_loader))

# mlm_sample の出力が(inputs, labels) の２つなので
inputs, labels = batch

print(f"input[0]:\n{inputs[0]}")
print(f"label[0]:\n{labels[0]}")

input[0]:
tensor([   1,  240, 5193, 3878, 5113,    4, 7998,  210, 1738, 4121,    4, 9087,
        7765, 9265, 4148, 3749,    2,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0])
label[0]:
tensor([-100, -100, -100, -100, -100, 7541, -100, -100, -100, -100, 3913, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100])
